In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain.agents import create_agent

In [ ]:

load_dotenv()

# Avoid hardcoding API keys in your script.
api_key = os.getenv("GEMINI_API_KEY")

# 1. Initialize the modern Gemini model wrapper
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    google_api_key=api_key,
    temperature=0.2
)

# 2. Invoke the model (LangChain uses .invoke() instead of calling the object directly)
response = llm.invoke("Write a poem on my love for dosa")

# 3. Print the text content of the message response
print(response.content)

In [ ]:
# =====================================================================
# DATABASE CONNECTION (CRITICAL SECURITY STEP: USE A READ-ONLY USER)
# =====================================================================

db_user = "root"
db_password = os.getenv("DB_PASSWORD")  # Replace with your actual password or use environment variables for better security
db_host = "localhost"
db_name = "atliq_tshirts"

db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}",sample_rows_in_table_info=3)

print(db.table_info)

In [ ]:
# =====================================================================
# TOOLKIT SETUP
# =====================================================================
# This automatically creates tools like:
# - sql_db_list_tables (lists visible tables)
# - sql_db_schema (fetches columns/types for a table safely)
# - sql_db_query_checker (uses LLM to inspect code for syntax errors before running)
# - sql_db_query (executes read-only commands)
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()

In [ ]:
# =====================================================================
# AGENT CREATION
# =====================================================================
agent_executor = create_agent(
    model=llm,
    tools=tools,
)

In [ ]:
# =====================================================================
# EXECUTION EXAMPLES
# =====================================================================
# Safe Query
safe_prompt = "How many t-shirts do we have left for nike in extra small size and white color?"
response = agent_executor.invoke({"messages": [("user", safe_prompt)]})
print("Answer:", response["messages"][-1].content)

In [20]:
safe_prompt = "How much is the price of the inventory for all small size t-shirts?"
response = agent_executor.invoke({"messages": [("user", safe_prompt)]})
print("Answer:", response["messages"][-1].content)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 38.279965237s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT